# Data Quality Checks and Preprocessing Decisions

**CS316 Electricity Theft Detection, Member 1 (data pipeline)**

This notebook shows the checks run on the two raw datasets and the
preprocessing decision each check led to. The cleaning itself is done by
the project code in `src/data_sgcc.py` and `src/data_oedi.py`
(run with `python -m src.prepare_data`).

Scope: data quality only (structure, missing values, outliers, labels).
Visual exploration of usage patterns is in the EDA notebook (Member 2).

## 0. Setup

Upload the raw files to Google Drive with this layout, then run the cells:

```
MyDrive/cs316-data/raw/sgcc/data.csv
MyDrive/cs316-data/raw/oedi/df.csv
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# In Colab: connect Google Drive. Elsewhere: set DATA_DIR to the local folder.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/cs316-data/raw"
except ImportError:
    DATA_DIR = "../data/raw"

SGCC_FILE = f"{DATA_DIR}/sgcc/data.csv"
OEDI_FILE = f"{DATA_DIR}/oedi/df.csv"
pd.set_option("display.max_columns", 15)

# Part A: SGCC dataset

## A1. First look

In [ ]:
# Read the readings as float32 instead of float64 to halve memory.
header = pd.read_csv(SGCC_FILE, nrows=0).columns
dtypes = {c: np.float32 for c in header if c not in ("CONS_NO", "FLAG")}
sgcc = pd.read_csv(SGCC_FILE, dtype=dtypes)

print("shape (customers, columns):", sgcc.shape)
print("memory (MB):", round(sgcc.memory_usage(deep=True).sum() / 1e6, 1))
sgcc.head()

In [ ]:
sgcc.info()

**Result:** one row per customer (`CONS_NO`), a label (`FLAG`: 1 = theft,
0 = normal) and one column per day of usage in kWh.

## A2. Are the day columns in date order?

In [ ]:
day_cols = sgcc.columns.drop(["CONS_NO", "FLAG"])
print("first 8 day columns:", list(day_cols[:8]))

dates = pd.to_datetime(day_cols, format="%Y/%m/%d")
print("in date order:", dates.is_monotonic_increasing)
print("range:", dates.min().date(), "to", dates.max().date())

all_days = pd.date_range(dates.min(), dates.max(), freq="D")
print("days in range:", len(all_days), "| day columns in file:", len(day_cols))
print("missing day(s):", [d.date() for d in all_days.difference(dates)])

**Decision:** the columns are sorted as text (`2014/1/1`, `2014/1/10`, ...),
not by date. We sort them by real date and add the one missing day as an
empty column, so the time series has no hole. It is filled with the other
gaps.

## A3. Missing values

In [ ]:
readings = sgcc[day_cols]
total_missing = readings.isnull().sum().sum()
print(f"missing readings: {total_missing:,} of {readings.size:,} ({total_missing / readings.size:.1%})")
print("customers with at least one missing day:", readings.isnull().any(axis=1).sum())
print("customers with NO readings at all:", readings.isnull().all(axis=1).sum())

In [ ]:
# Share of missing days per customer.
missing_share = readings.isnull().mean(axis=1)
missing_share.describe()

In [ ]:
# Is missing data more common among theft customers?
missing_share.groupby(sgcc["FLAG"]).describe()

**Decisions:**
- Customers with no readings at all are removed.
- Every customer has some gaps, so rows cannot be dropped. Gaps are filled.
- Before filling, each customer's share of missing days is saved as a
  column (`missing_share`), since it differs between theft and normal
  customers and may be useful as a feature.

## A4. Duplicates and invalid values

In [ ]:
print("duplicate rows:", sgcc.duplicated().sum())
print("duplicate customer IDs:", sgcc["CONS_NO"].duplicated().sum())
print("negative readings:", (readings < 0).sum().sum())

**Decision:** no duplicates and no negative readings, so no action needed.

## A5. Extreme values

In [ ]:
# describe() on all real daily readings (missing days excluded).
all_values = pd.Series(readings.to_numpy().ravel()).dropna()
all_values.describe(percentiles=[0.25, 0.5, 0.75, 0.99, 0.999])

**Result:** 75% of daily readings are small, but the maximum is hundreds of
thousands of kWh, which is not realistic for one day. These look like
recording errors.

We cap them with the IQR rule, separately for each customer, because a
large business and a small home have different normal levels:
`upper limit = Q3 + k x IQR`. Readings above the limit are replaced by the
limit (not deleted). Low values and zeros are never changed.

### Choosing the IQR settings

We compared the usual `k = 1.5` with `k = 3` (Tukey's "extreme outlier"
fence), and computing Q1/Q3 on all days vs on non-zero days only.

In [ ]:
def iqr_caps(r, k, nonzero_only):
    """True where a reading is above the customer's IQR upper limit."""
    base = r.where(r > 0) if nonzero_only else r
    q1, q3 = base.quantile(0.25, axis=1), base.quantile(0.75, axis=1)
    iqr = q3 - q1
    upper = (q3 + k * iqr).where(iqr > 0, np.inf)  # IQR = 0: skip customer
    return r.gt(upper, axis=0), int((iqr == 0).sum())

r = readings[~readings.isnull().all(axis=1)]
real = r.notnull().sum().sum()
rows = []
for nonzero_only in (False, True):
    for k in (1.5, 3.0):
        capped, skipped = iqr_caps(r, k, nonzero_only)
        n = capped.sum().sum()
        rows.append({"k": k, "Q1/Q3 from": "non-zero days" if nonzero_only else "all days",
                     "readings capped": n, "% of real readings": round(100 * n / real, 2),
                     "customers skipped (IQR=0)": skipped})
pd.DataFrame(rows)

**Decision:** `k = 3` on non-zero days.
- Daily electricity use is skewed (many low days, some naturally high
  days). With `k = 1.5`, about 4% of all readings were capped, mostly normal
  high-usage days, not errors. `k = 3` caps about 1.2%.
- For customers with many zero days, Q1 and Q3 on all days are close to 0,
  so their normal usage would be capped. Using non-zero days avoids this.
- Customers whose IQR is still 0 are skipped.

### Example: one customer before and after capping

In [ ]:
# The customer with the single largest reading in the dataset.
row = r.max(axis=1).idxmax()
series = r.loc[row].copy()
series.index = pd.to_datetime(series.index, format="%Y/%m/%d")
series = series.sort_index()

nonzero = series[series > 0]
q1, q3 = nonzero.quantile(0.25), nonzero.quantile(0.75)
limit = q3 + 3 * (q3 - q1)
capped = series.clip(upper=limit)
print(f"customer {sgcc.loc[row, 'CONS_NO']}: median {series.median():.1f} kWh, "
      f"max {series.max():,.0f} kWh, limit {limit:.1f} kWh")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(series, lw=0.8)
axes[0].axhline(limit, color="red", ls="--", label=f"IQR limit ({limit:.1f})")
axes[0].set_yscale("log")
axes[0].set_title("Before capping (log scale)")
axes[0].legend()
axes[1].plot(capped, lw=0.8)
axes[1].axhline(limit, color="red", ls="--")
axes[1].set_title("After capping")
for ax in axes:
    ax.set_ylabel("kWh per day")
plt.tight_layout()
plt.show()

## A6. How to fill the gaps

In [ ]:
# Find a customer with a gap of 5 to 15 days in the middle of their series.
ordered = r.copy()
ordered.columns = pd.to_datetime(ordered.columns, format="%Y/%m/%d")
ordered = ordered.sort_index(axis=1)

def middle_gap(s, low=5, high=15):
    """Return (start, end) positions of a middle gap of the given length, or None."""
    is_na = s.isnull().to_numpy()
    i = 0
    while i < len(is_na):
        if is_na[i]:
            j = i
            while j < len(is_na) and is_na[j]:
                j += 1
            if 0 < i and j < len(is_na) and low <= j - i <= high:
                return i, j
            i = j
        else:
            i += 1
    return None

for idx in ordered.index:
    gap = middle_gap(ordered.loc[idx])
    if gap:
        break
s = ordered.loc[idx]
start, end = gap
window = s.iloc[max(start - 15, 0): end + 15]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(window.ffill(), "--", color="orange", label="forward fill")
ax.plot(window.interpolate(limit_area="inside"), "-", color="green", label="interpolation")
ax.plot(window, "o", color="black", ms=4, label="real readings")
ax.axvspan(s.index[start], s.index[end - 1], color="grey", alpha=0.2, label="missing days")
ax.set_ylabel("kWh per day")
ax.set_title(f"Filling a {end - start}-day gap")
ax.legend()
plt.show()

**Decision:**
- **Middle gaps:** linear interpolation. Usage changes gradually, and
  forward fill creates a flat line followed by a sudden jump, which could
  look like a sudden change in usage (a theft clue).
- **Start of a series:** backward fill. **End of a series:** forward fill.
  Interpolation needs known values on both sides, so it cannot fill these.
- Outliers are capped **before** filling, so extreme values do not spread
  into filled days.
- Each customer is filled using only their own readings (no leakage).

## A7. Labels and split

In [ ]:
print(sgcc["FLAG"].value_counts())
print(sgcc["FLAG"].value_counts(normalize=True).round(3))

**Decision:** only about 8.5% of customers are labelled as theft, so the
80/20 train/test split is **stratified** by `FLAG` (both sets keep 8.5%
theft), with `random_state = 42`. Each row is one customer, so no customer
appears in both sets.

# Part B: OEDI dataset

## B1. First look

In [ ]:
oedi = pd.read_csv(OEDI_FILE)
print("shape:", oedi.shape)
oedi.head()

In [ ]:
oedi.info()

In [ ]:
oedi.describe().T

In [ ]:
print("column '0' equals the row number:", (oedi["0"] == np.arange(len(oedi))).all())
print("missing values per column:")
oedi.isnull().sum()

**Result and decision:** each row is one hour of usage for one building.
Column `0` is only the row number, so it is dropped. There are no missing
values, so no filling is needed.

## B2. Building types and labels

In [ ]:
oedi["Class"].value_counts()

In [ ]:
oedi["theft"].value_counts()

**Decisions:**
- 15 rows have building type `"0"`; they are broken and removed.
- The label has 7 values. We create a 0/1 `theft` label (any theft type = 1)
  and keep the original as `theft_type` for error analysis.

## B3. Structure of the file

In [ ]:
clean = oedi[oedi["Class"] != "0"].reset_index(drop=True)
recording = (clean["Class"] != clean["Class"].shift()).cumsum()
sizes = recording.value_counts()
print("number of recordings:", recording.nunique())
print("rows per recording:", sizes.unique())
print("recordings per building type:", clean.groupby("Class").size().div(8760).unique())

**Result:** the file is 64 one-year recordings (8,760 hours = 365 x 24):
16 building types x 4 recordings each. Comparing the 4 recordings of each
type showed they are similar but not copies (almost no identical hours),
and theft was added at different times in each.

## B4. Why a random split would leak

In [ ]:
elec = "Electricity:Facility [kW](Hourly)"
first = clean[recording == 1].reset_index(drop=True)
start = first.index[first["theft"] != "Normal"][0]
window = first.iloc[max(start - 48, 0): start + 72]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(window.index, window[elec], color="black", lw=1)
is_theft = window["theft"] != "Normal"
ax.fill_between(window.index, 0, window[elec].max(), where=is_theft,
                color="red", alpha=0.2, label="theft hours")
ax.set_xlabel("hour of the year")
ax.set_ylabel("kW")
ax.set_title("Theft lasts for a stretch of neighbouring hours")
ax.legend()
plt.show()

runs = (clean["theft"] != clean["theft"].shift()).cumsum()
lengths = clean.groupby(runs)["theft"].agg(["first", "size"])
print("theft stretch length (hours):")
lengths.loc[lengths["first"] != "Normal", "size"].describe()

**Decision:** theft lasts for stretches of neighbouring hours (typically
24). If rows were split at random, hours from the same theft stretch would
be in both training and test sets, and scores would be too optimistic.
Instead we split **by recording**: for each building type, 1 of its 4
recordings (chosen with seed 42) goes to the test set and 3 to training
(75/25). Every building type is seen in training, and test recordings are
never seen during training.

# Summary of preprocessing decisions

| Dataset | Issue found | Decision |
|---|---|---|
| SGCC | Day columns sorted as text; one day missing | Sort by date; add the missing day |
| SGCC | 5 customers with no readings | Remove |
| SGCC | 25.6% readings missing | Save `missing_share`; interpolate middle gaps, backward/forward fill edges |
| SGCC | Extreme readings (up to 800,003 kWh) | Cap per customer: IQR, k = 3, on non-zero days |
| SGCC | 8.5% theft | Stratified 80/20 split by customer |
| OEDI | Row-number column | Drop |
| OEDI | 15 broken rows | Remove |
| OEDI | 7 label values | Binary `theft` + `theft_type` |
| OEDI | Theft in stretches of neighbouring hours | Split by recording (1 of 4 per type to test) |
| Both | Memory | Readings stored as float32 |